In [8]:
import os 
from dotenv import load_dotenv

load_dotenv()

# OpenAI client reads OPENAI_API_KEY (not OPEN_API_KEY)
_openai_key = os.getenv("OPENAI_API_KEY") or os.getenv("OPEN_API_KEY")
if _openai_key:
    os.environ["OPENAI_API_KEY"] = _openai_key

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"



In [ ]:
## create a datapoints 
## that;s how i instertthe data in langsmith


from langsmith import Client

client = Client()

## Define the dataset - these are my test data 
dataset_name = "MychatBot"

dataset=client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=[

    {"inputs": {"question": "What is this?"}, "outputs": {"answer": "This is a chatbot designed to assist you with your queries."}},
    {"inputs": {"question": "Who created you?"}, "outputs": {"answer": "I was created by a team of developers to help answer questions."}},
    {"inputs": {"question": "How can you help me?"}, "outputs": {"answer": "I can provide information, answer questions, and help you with various tasks."}},
    {"inputs": {"question": "What technologies do you use?"}, "outputs": {"answer": "I use natural language processing and machine learning to understand and respond to your questions."}},
    {"inputs": {"question": "Are you always available?"}, "outputs": {"answer": "Yes, I am available 24/7 to assist you."}},

    ]
)



{'example_ids': ['0dedbade-b0c7-4200-962d-2db37e671d25',
  '8fe5da82-636a-432c-8653-fd199735e5d1',
  '178ca48c-5b7e-422a-a618-900f05d5fce6',
  '762f8c38-6fd1-4a67-ab12-290a3d9408b1',
  '7b1fc147-d9d0-47f1-a9c5-162f772cc7e5'],
 'count': 5,
 'as_of': '2026-03-28T20:16:04.945657179Z'}

In [21]:
### Define metrics (LLm as a judge)

import openai

from langsmith import wrappers

openai_client = wrappers.wrap_openai(openai.OpenAI())

instruction = (
    "You grade chatbot answers against a reference. "
    "Answer YES if the chatbot response is correct and helpful for the question, "
    "including when it is semantically equivalent to the reference or adds reasonable detail. "
    "Answer NO only if it is wrong, misleading, refuses unreasonably, or misses the intent."
)

def _pred_answer(outputs: dict) -> str:
    return str(outputs.get("answer") or outputs.get("response") or "")


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = (
        f"Input Question: {inputs['question']}\n"
        f"Chatbot Answer: {_pred_answer(outputs)}\n"
        f"Reference Answer: {reference_outputs['answer']}\n\n"
        "Reply with only YES or NO."
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user", "content": user_content},
        ],
    ).choices[0].message.content

    text = (response or "").strip().upper()
    return text.startswith("Y")



In [22]:
# Concision: 0–1 score — 1.0 if as short or shorter than reference, else decays as the answer gets wordier
# (A strict bool `len(pred) <= len(ref)` is almost always False for real models vs short gold answers.)


def concision(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    pred = str(outputs.get("answer") or outputs.get("response") or "")
    ref = str(reference_outputs.get("answer", ""))
    pred_n = max(len(pred.split()), 1)
    ref_n = max(len(ref.split()), 1)
    return float(min(1.0, ref_n / pred_n))

In [23]:
### how to run the evaluation 

default_instruction = "Please evaluate the answers generated by the chatbot for correctness and helpfulness."

def my_app(question:str, model:str = "gpt-4o-mini", instruction:str = default_instruction)-> str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content



In [24]:
### Call my app for every datapoints 

def ls_target(inputs: dict) -> dict:
    # Use "answer" so it matches dataset reference_outputs and evaluators
    return {"answer": my_app(inputs["question"])}

In [25]:
experiment_results = client.evaluate(
    ls_target,
    data= dataset_name,
    evaluators = [correctness, concision],
    experiment_prefix = 'openai-4o-mini-chatbot'
)

View the evaluation results for experiment: 'openai-4o-mini-chatbot-bdeb98a4' at:
https://smith.langchain.com/o/625709e1-aba8-489e-9216-414289067b1e/datasets/b8f87aeb-de1e-4e53-8ff9-b443192f2569/compare?selectedSessions=3ab2365a-719e-43d2-9e8d-ef4354940b57




5it [00:39,  7.93s/it]
